<a href="https://colab.research.google.com/github/ndthien90/HSK_Audio/blob/main/OmniVoice_Thien90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌍 OmniVoice — Gradio Interface
**Zero-shot TTS with voice cloning & voice design across 600+ languages**

Run each cell in order. Gradio UI launches at the bottom.

In [ ]:
# Cell 1: Install dependencies
!pip install -q omnivoice gradio
!pip install -q torchaudio --extra-index-url https://download.pytorch.org/whl/cu128

In [ ]:
# Cell 2: Load model
import torch
import torchaudio
import gradio as gr
import tempfile
from omnivoice import OmniVoice

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')

print('Loading OmniVoice model (first time downloads ~3-5 GB)...')
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=device, dtype=dtype)
print('Model loaded!')

Device: cuda:0
GPU: Tesla T4 (15.6 GB)
Loading OmniVoice model (first time downloads ~3-5 GB)...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Model loaded!


In [ ]:
import gradio as gr
import tempfile
import torch
import torchaudio
import torchaudio.functional as F
import random
import zipfile
import os
import re

# (Giả sử mô hình OmniVoice của bạn đã được khởi tạo ở đây)
# model = OmniVoiceModel()

NONVERBAL_TAGS = [
    '[laughter]', '[sigh]', '[sniff]',
    '[question-en]', '[surprise-ah]', '[dissatisfaction-hnn]', '[confirmation-en]',
]

DESIGN_PRESETS = {
    'Custom (type your own)': '',
    'Female, British accent': 'female, british accent',
    'Male, American accent': 'male, american accent',
    'Female, high pitch, young': 'female, high pitch, young adult',
    'Male, deep, elderly': 'male, very low pitch, elderly',
    'Female, whisper': 'female, whisper',
    'Male, Australian accent': 'male, australian accent',
    'Female, Indian accent': 'female, indian accent',
    'Male, fast, energetic': 'male, high pitch, young adult',
}

def generate(text, mode, clone_sub_mode, odd_voice, even_voice, naming_rule, enable_merge, merge_count,
             batch_conv_output_mode,
             ref_audio_1, ref_text_1, ref_audio_2, ref_text_2, ref_audio_3, ref_text_3,
             design_preset, custom_instruct, num_steps, speed,
             pause_duration, audio_quality, audio_format):

    if not text.strip():
        return None, None, 'Vui lòng nhập văn bản.'
    try:
        kwargs = dict(num_step=int(num_steps))
        global_speed = float(speed)
        pause_sec = float(pause_duration)
        fmt = audio_format.lower()

        # Xác định Tần số lấy mẫu (Sample Rate) mục tiêu
        target_sr = 24000
        if '16kHz' in audio_quality: target_sr = 16000
        elif '8kHz' in audio_quality: target_sr = 8000

        # ==========================================
        # 1. XỬ LÝ: CHẾ ĐỘ BATCH CONVERSATION (TẠO HÀNG LOẠT HỘI THOẠI)
        # ==========================================
        if mode == 'Batch Conversation':
            # Phân tách bằng dấu # hoặc dòng trống (hỗ trợ cả 2 định dạng)
            conversations = [c.strip() for c in re.split(r'\n\s*\n|\n?\s*#\s*\n?', text.strip()) if c.strip()]

            if not conversations:
                return None, None, 'Không tìm thấy đoạn hội thoại nào hợp lệ.'

            temp_dir = tempfile.mkdtemp()
            zip_filename = os.path.join(temp_dir, "batch_conversations.zip")

            with zipfile.ZipFile(zip_filename, 'w') as zipf:
                for conv_idx, conv_text in enumerate(conversations):
                    # Tìm tag [HTx] để đặt tên nếu có, nếu không thì dùng số thứ tự
                    ht_match = re.search(r'\[HT(\d+)\]', conv_text)
                    ht_num = ht_match.group(1) if ht_match else str(conv_idx + 1)

                    is_normal_mode = (batch_conv_output_mode == 'Bình thường')

                    if is_normal_mode:
                        ht_folder = f"HT{ht_num}"
                        audio_folder = f"{ht_folder}/audio"
                        txt_filename = f"{ht_folder}/HT{ht_num}.txt"
                        zipf.writestr(txt_filename, conv_text.encode('utf-8'))

                    lines = [l.strip() for l in conv_text.split('\n') if l.strip()]
                    conv_audio_segments = []
                    spoken_line_idx = 0

                    for line_idx, line in enumerate(lines):
                        if line.startswith('A:'):
                            speak_text = line[2:].split('|')[0].strip()
                            current_voice_choice = 'Clone 1'
                        elif line.startswith('B:'):
                            speak_text = line[2:].split('|')[0].strip()
                            current_voice_choice = 'Clone 2'
                        elif re.match(r'^\[HT\d*\]', line) or line.startswith('[TL]'):
                            continue # 🔴 Bỏ qua câu hỏi người thứ 3 và đáp án trong chế độ này
                        else:
                            speak_text = line.split('|')[0].strip()
                            current_voice_choice = odd_voice if (spoken_line_idx + 1) % 2 != 0 else even_voice

                        if not speak_text: continue

                        spoken_line_idx += 1
                        line_num = spoken_line_idx

                        kwargs.pop('ref_audio', None)
                        kwargs.pop('ref_text', None)
                        kwargs.pop('instruct', None)
                        kwargs.pop('seed', None)

                        if current_voice_choice == 'Clone 1':
                            if not ref_audio_1: return None, None, f'Lỗi: Thiếu audio Clone 1 (HT{ht_num}, câu {line_num})'
                            kwargs['ref_audio'] = ref_audio_1
                            if ref_text_1.strip(): kwargs['ref_text'] = ref_text_1
                        elif current_voice_choice == 'Clone 2':
                            if not ref_audio_2: return None, None, f'Lỗi: Thiếu audio Clone 2 (HT{ht_num}, câu {line_num})'
                            kwargs['ref_audio'] = ref_audio_2
                            if ref_text_2.strip(): kwargs['ref_text'] = ref_text_2

                        kwargs['text'] = speak_text
                        kwargs['speed'] = global_speed

                        audio = model.generate(**kwargs)
                        audio_tensor = audio[0] if isinstance(audio[0], torch.Tensor) else torch.tensor(audio[0])
                        if audio_tensor.dim() == 1: audio_tensor = audio_tensor.unsqueeze(0)

                        if target_sr != 24000:
                            audio_tensor = F.resample(audio_tensor, orig_freq=24000, new_freq=target_sr)

                        if is_normal_mode:
                            out_filename = f"{audio_folder}/output_{line_num}.{fmt}"
                            tmp_wav_path = os.path.join(temp_dir, f"tmp_{ht_num}_{line_num}.{fmt}")
                            torchaudio.save(tmp_wav_path, audio_tensor.cpu(), target_sr)
                            zipf.write(tmp_wav_path, arcname=out_filename)

                        # Chèn khoảng lặng trước khi thêm câu mới (nếu không phải câu đầu)
                        if len(conv_audio_segments) > 0 and pause_sec > 0:
                            silence = torch.zeros((1, int(target_sr * pause_sec)))
                            conv_audio_segments.append(silence)

                        conv_audio_segments.append(audio_tensor)

                    if conv_audio_segments:
                        final_merged_audio = torch.cat(conv_audio_segments, dim=-1)
                        merge_filename = f"{audio_folder}/merge.{fmt}" if is_normal_mode else f"HT{ht_num}.{fmt}"

                        tmp_merge_path = os.path.join(temp_dir, f"tmp_merge_{ht_num}.{fmt}")
                        torchaudio.save(tmp_merge_path, final_merged_audio.cpu(), target_sr)
                        zipf.write(tmp_merge_path, arcname=merge_filename)

            return None, zip_filename, f'✅ Đã tạo ZIP chứa {len(conversations)} đoạn hội thoại (bỏ qua người thứ 3).'

        # ==========================================
        # 1.5 XỬ LÝ: CHẾ ĐỘ HSK CONVERSATION (CÓ NGƯỜI THỨ 3 HỎI)
        # ==========================================
        elif mode == 'HSK Conversation (3 Voices)':
            conversations = [c.strip() for c in re.split(r'\n?\s*#\s*\n?', text.strip()) if c.strip()]

            if not conversations:
                return None, None, 'Không tìm thấy đoạn hội thoại HSK nào hợp lệ.'

            temp_dir = tempfile.mkdtemp()
            zip_filename = os.path.join(temp_dir, "hsk_conversations.zip")

            with zipfile.ZipFile(zip_filename, 'w') as zipf:
                for conv_idx, conv_text in enumerate(conversations):
                    ht_match = re.search(r'\[HT(\d+)\]', conv_text)
                    ht_num = ht_match.group(1) if ht_match else str(conv_idx + 1)

                    is_normal_mode = (batch_conv_output_mode == 'Bình thường')

                    if is_normal_mode:
                        ht_folder = f"HT{ht_num}"
                        audio_folder = f"{ht_folder}/audio"
                        txt_filename = f"{ht_folder}/HT{ht_num}.txt"
                        zipf.writestr(txt_filename, conv_text.encode('utf-8'))

                    lines = [l.strip() for l in conv_text.split('\n') if l.strip()]
                    conv_audio_segments = []

                    for line_idx, line in enumerate(lines):
                        speak_text = ""
                        current_voice = 1

                        if line.startswith('A:'):
                            speak_text = line[2:].strip()
                            current_voice = 1
                        elif line.startswith('B:'):
                            speak_text = line[2:].strip()
                            current_voice = 2
                        elif re.match(r'^\[HT\d*\]', line):
                            speak_text = re.sub(r'^\[HT\d*\]', '', line).strip()
                            current_voice = 3
                        elif line.startswith('[TL]'):
                            continue
                        else:
                            continue

                        if not speak_text: continue

                        kwargs.pop('ref_audio', None)
                        kwargs.pop('ref_text', None)
                        kwargs.pop('instruct', None)
                        kwargs.pop('seed', None)

                        if current_voice == 1:
                            if not ref_audio_1: return None, None, f'Lỗi: Thiếu audio Clone 1 (HT{ht_num})'
                            kwargs['ref_audio'] = ref_audio_1
                            if ref_text_1.strip(): kwargs['ref_text'] = ref_text_1
                        elif current_voice == 2:
                            if not ref_audio_2: return None, None, f'Lỗi: Thiếu audio Clone 2 (HT{ht_num})'
                            kwargs['ref_audio'] = ref_audio_2
                            if ref_text_2.strip(): kwargs['ref_text'] = ref_text_2
                        elif current_voice == 3:
                            if not ref_audio_3: return None, None, f'Lỗi: Thiếu audio Clone 3 (Người hỏi HT{ht_num})'
                            kwargs['ref_audio'] = ref_audio_3
                            if ref_text_3.strip(): kwargs['ref_text'] = ref_text_3

                        kwargs['text'] = speak_text
                        kwargs['speed'] = global_speed

                        audio = model.generate(**kwargs)
                        audio_tensor = audio[0] if isinstance(audio[0], torch.Tensor) else torch.tensor(audio[0])
                        if audio_tensor.dim() == 1: audio_tensor = audio_tensor.unsqueeze(0)

                        if target_sr != 24000:
                            audio_tensor = F.resample(audio_tensor, orig_freq=24000, new_freq=target_sr)

                        if is_normal_mode:
                            out_filename = f"{audio_folder}/output_{line_idx+1}.{fmt}"
                            tmp_wav_path = os.path.join(temp_dir, f"tmp_{ht_num}_{line_idx+1}.{fmt}")
                            torchaudio.save(tmp_wav_path, audio_tensor.cpu(), target_sr)
                            zipf.write(tmp_wav_path, arcname=out_filename)

                        if len(conv_audio_segments) > 0:
                            if current_voice == 3:
                                silence = torch.zeros((1, int(target_sr * 1.0))) # Chờ 1 giây trước khi hỏi
                            elif pause_sec > 0:
                                silence = torch.zeros((1, int(target_sr * pause_sec)))
                            else:
                                silence = None

                            if silence is not None:
                                conv_audio_segments.append(silence)

                        conv_audio_segments.append(audio_tensor)

                    if conv_audio_segments:
                        final_merged_audio = torch.cat(conv_audio_segments, dim=-1)
                        merge_filename = f"{audio_folder}/merge.{fmt}" if is_normal_mode else f"HT{ht_num}.{fmt}"

                        tmp_merge_path = os.path.join(temp_dir, f"tmp_merge_{ht_num}.{fmt}")
                        torchaudio.save(tmp_merge_path, final_merged_audio.cpu(), target_sr)
                        zipf.write(tmp_merge_path, arcname=merge_filename)

            return None, zip_filename, f'✅ Đã tạo ZIP chứa {len(conversations)} hội thoại HSK.'

        # ==========================================
        # 2. XỬ LÝ: CHẾ ĐỘ TẠO ZIP TỪNG DÒNG (BATCH CLONING)
        # ==========================================
        elif mode == 'Voice Cloning' and clone_sub_mode == 'Line-by-Line (Batch ZIP)':
            lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
            if not lines:
                return None, None, 'Không tìm thấy dòng văn bản hợp lệ nào.'

            temp_dir = tempfile.mkdtemp()
            zip_filename = os.path.join(temp_dir, "omnivoice_batch.zip")

            all_generated_audios = []

            with zipfile.ZipFile(zip_filename, 'w') as zipf:

                for i, line in enumerate(lines):
                    line_num = i + 1
                    current_voice_choice = odd_voice if line_num % 2 != 0 else even_voice

                    kwargs.pop('ref_audio', None)
                    kwargs.pop('ref_text', None)

                    if current_voice_choice == 'Clone 1':
                        if not ref_audio_1: return None, None, f'Lỗi: Thiếu audio Clone 1 (dòng {line_num})'
                        kwargs['ref_audio'] = ref_audio_1
                        if ref_text_1.strip(): kwargs['ref_text'] = ref_text_1
                    elif current_voice_choice == 'Clone 2':
                        if not ref_audio_2: return None, None, f'Lỗi: Thiếu audio Clone 2 (dòng {line_num})'
                        kwargs['ref_audio'] = ref_audio_2
                        if ref_text_2.strip(): kwargs['ref_text'] = ref_text_2

                    kwargs['text'] = line
                    kwargs['speed'] = global_speed

                    audio = model.generate(**kwargs)
                    audio_tensor = audio[0] if isinstance(audio[0], torch.Tensor) else torch.tensor(audio[0])
                    if audio_tensor.dim() == 1: audio_tensor = audio_tensor.unsqueeze(0)

                    if target_sr != 24000:
                        audio_tensor = F.resample(audio_tensor, orig_freq=24000, new_freq=target_sr)

                    all_generated_audios.append(audio_tensor)

                    if naming_rule == 'Sequential (output_1)':
                        filename = f"output_{line_num}.{fmt}"
                    else:
                        clean_str = re.sub(r'[\\/*?:"<>|]', '', line).strip()[:300]
                        if not clean_str: clean_str = f"output_{line_num}"
                        filename = f"{clean_str}.{fmt}"

                    wav_path = os.path.join(temp_dir, filename)
                    torchaudio.save(wav_path, audio_tensor.cpu(), target_sr)
                    zipf.write(wav_path, arcname=filename)

                num_merged_files = 0
                if enable_merge:
                    merge_n = int(merge_count)
                    chunks = [all_generated_audios[i:i + merge_n] for i in range(0, len(all_generated_audios), merge_n)]
                    num_merged_files = len(chunks)

                    for chunk_idx, chunk_tensors in enumerate(chunks):
                        merged_segments = []
                        for idx, tensor in enumerate(chunk_tensors):
                            merged_segments.append(tensor)
                            if pause_sec > 0 and idx < len(chunk_tensors) - 1:
                                silence = torch.zeros((1, int(target_sr * pause_sec)))
                                merged_segments.append(silence)

                        if merged_segments:
                            final_merged_audio = torch.cat(merged_segments, dim=-1)
                            merge_filename = f"merge{chunk_idx + 1}.{fmt}"
                            merge_path = os.path.join(temp_dir, merge_filename)

                            torchaudio.save(merge_path, final_merged_audio.cpu(), target_sr)
                            zipf.write(merge_path, arcname=merge_filename)

            msg = f'✅ Đã tạo ZIP ({len(lines)} file lẻ'
            if enable_merge:
                msg += f' + {num_merged_files} file ghép'
            msg += f' định dạng {audio_format.upper()}).'

            return None, zip_filename, msg

        # ==========================================
        # 3. XỬ LÝ: CHẾ ĐỘ CONVERSATION (THEO DÒNG CHẴN LẺ - SINGLE FILE)
        # ==========================================
        elif mode == 'Conversation':
            lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
            if not lines:
                return None, None, 'Không tìm thấy dòng văn bản hợp lệ nào.'

            audio_segments = []

            for i, line in enumerate(lines):
                line_num = i + 1
                current_voice_choice = odd_voice if line_num % 2 != 0 else even_voice

                kwargs.pop('ref_audio', None)
                kwargs.pop('ref_text', None)
                kwargs.pop('instruct', None)
                kwargs.pop('seed', None)

                if current_voice_choice == 'Clone 1':
                    if not ref_audio_1: return None, None, f"Lỗi: Thiếu audio Clone 1 (cho dòng {line_num})."
                    kwargs['ref_audio'] = ref_audio_1
                    if ref_text_1.strip(): kwargs['ref_text'] = ref_text_1
                elif current_voice_choice == 'Clone 2':
                    if not ref_audio_2: return None, None, f"Lỗi: Thiếu audio Clone 2 (cho dòng {line_num})."
                    kwargs['ref_audio'] = ref_audio_2
                    if ref_text_2.strip(): kwargs['ref_text'] = ref_text_2

                kwargs['text'] = line
                kwargs['speed'] = global_speed

                audio = model.generate(**kwargs)
                audio_tensor = audio[0] if isinstance(audio[0], torch.Tensor) else torch.tensor(audio[0])
                if audio_tensor.dim() == 1: audio_tensor = audio_tensor.unsqueeze(0)

                audio_segments.append(audio_tensor)

                if pause_sec > 0:
                    silence = torch.zeros((1, int(24000 * pause_sec)))
                    audio_segments.append(silence)

            if not audio_segments: return None, None, 'Lỗi trong quá trình tạo Conversation.'

            if pause_sec > 0:
                final_audio_tensor = torch.cat(audio_segments[:-1], dim=-1)
            else:
                final_audio_tensor = torch.cat(audio_segments, dim=-1)

            if target_sr != 24000:
                final_audio_tensor = F.resample(final_audio_tensor, orig_freq=24000, new_freq=target_sr)

            tmp = tempfile.NamedTemporaryFile(suffix=f'.{fmt}', delete=False)
            torchaudio.save(tmp.name, final_audio_tensor.cpu(), target_sr)
            duration = final_audio_tensor.shape[-1] / target_sr

            return tmp.name, None, f'Generated {duration:.1f}s | Mode: Conversation | {audio_format.upper()}'

        # ==========================================
        # 4. XỬ LÝ: CHẾ ĐỘ AUTO VOICE, DESIGN VÀ SINGLE CLONING
        # ==========================================
        else:
            lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
            if not lines: return None, None, 'Vui lòng nhập văn bản.'

            audio_segments = []
            kwargs['speed'] = global_speed

            if mode == 'Voice Cloning':
                if ref_audio_1 is None: return None, None, 'Vui lòng tải lên Clone 1.'
                kwargs['ref_audio'] = ref_audio_1
                if ref_text_1.strip(): kwargs['ref_text'] = ref_text_1
            elif mode == 'Voice Design':
                instruct = custom_instruct.strip() if design_preset == 'Custom (type your own)' else DESIGN_PRESETS[design_preset]
                if not instruct: return None, None, 'Please enter a voice design instruction.'
                kwargs['instruct'] = instruct

            for line in lines:
                kwargs['text'] = line
                audio = model.generate(**kwargs)
                audio_tensor = audio[0] if isinstance(audio[0], torch.Tensor) else torch.tensor(audio[0])
                if audio_tensor.dim() == 1: audio_tensor = audio_tensor.unsqueeze(0)

                audio_segments.append(audio_tensor)

                if pause_sec > 0:
                    silence = torch.zeros((1, int(24000 * pause_sec)))
                    audio_segments.append(silence)

            if pause_sec > 0:
                final_audio_tensor = torch.cat(audio_segments[:-1], dim=-1)
            else:
                final_audio_tensor = torch.cat(audio_segments, dim=-1)

            if target_sr != 24000:
                final_audio_tensor = F.resample(final_audio_tensor, orig_freq=24000, new_freq=target_sr)

            tmp = tempfile.NamedTemporaryFile(suffix=f'.{fmt}', delete=False)
            torchaudio.save(tmp.name, final_audio_tensor.cpu(), target_sr)
            duration = final_audio_tensor.shape[-1] / target_sr

            return tmp.name, None, f'Generated {duration:.1f}s | Mode: {mode} | {audio_format.upper()}'

    except Exception as e:
        return None, None, f'Error: {e}'

def update_ui_visibility(mode, clone_sub_mode):
    is_cloning = mode == 'Voice Cloning'
    is_batch_zip = is_cloning and clone_sub_mode == 'Line-by-Line (Batch ZIP)'

    show_clones = mode in ['Voice Cloning', 'Conversation', 'Batch Conversation', 'HSK Conversation (3 Voices)']
    show_clone_3 = mode == 'HSK Conversation (3 Voices)'
    show_line_settings = is_batch_zip or mode in ['Conversation', 'Batch Conversation']
    show_batch_only_col = is_batch_zip
    show_batch_conv_only_col = mode in ['Batch Conversation', 'HSK Conversation (3 Voices)']

    return (
        gr.update(visible=show_clones),
        gr.update(visible=is_cloning),
        gr.update(visible=show_line_settings),
        gr.update(visible=show_batch_only_col),
        gr.update(visible=show_batch_conv_only_col),
        gr.update(visible=mode == 'Voice Design'),
        gr.update(visible=mode == 'Voice Design'),
        gr.update(visible=show_clone_3)
    )

def update_preset(preset):
    return gr.update(visible=preset == 'Custom (type your own)')

def insert_tag(text, tag):
    return text + ' ' + tag

css = '''
#header { text-align: center; padding: 16px 0 8px; }
#header h1 { font-size: 2.2em; margin-bottom: 4px; }
#header p { color: #888; }
.gen-btn { background: linear-gradient(135deg, #059669, #2563eb) !important; color: white !important; font-size: 1.05em !important; border:none; }
.batch-box { padding: 10px; background: #f0fdf4; border-radius: 8px; margin-top: 10px; border: 1px solid #bbf7d0;}
'''

with gr.Blocks(title='OmniVoice Ultra', theme=gr.themes.Soft(primary_hue='emerald', secondary_hue='blue'), css=css) as demo:
    gr.HTML("<div id='header'><h1>OmniVoice</h1><p>Voice Cloning &middot; Voice Design &middot; Batch Generator</p></div>")

    with gr.Row():
        with gr.Column(scale=3):
            text_input = gr.Textbox(label='Script / Text', placeholder='Line 1...\nLine 2...', lines=5)

            gr.Markdown('**Non-verbal tags:**')
            with gr.Row():
                for tag in NONVERBAL_TAGS:
                    b = gr.Button(tag, size='sm')
                    b.click(fn=insert_tag, inputs=[text_input, gr.State(tag)], outputs=text_input)

            mode_radio = gr.Radio(choices=['Auto Voice', 'Voice Cloning', 'Voice Design', 'Conversation', 'Batch Conversation', 'HSK Conversation (3 Voices)'], value='Voice Cloning', label='Generation mode')

            clone_sub_mode = gr.Radio(
                choices=['Single File (Merge all)', 'Line-by-Line (Batch ZIP)'],
                value='Single File (Merge all)',
                label='Cloning Mode Settings',
                visible=True
            )

            with gr.Row(visible=False, elem_classes="batch-box") as line_settings_row:
                with gr.Column():
                    odd_voice = gr.Dropdown(choices=['Clone 1', 'Clone 2'], value='Clone 1', label='Odd Lines (1, 3, 5...)')
                    even_voice = gr.Dropdown(choices=['Clone 1', 'Clone 2'], value='Clone 2', label='Even Lines (2, 4, 6...)')
                with gr.Column(visible=False) as batch_only_col:
                    naming_rule = gr.Radio(
                        choices=['Sequential (output_1)', 'Content-based (30 chars)'],
                        value='Content-based (30 chars)',
                        label='File Naming Rule'
                    )
                    enable_merge = gr.Checkbox(label='Tạo thêm file ghép (Generate Merged Files)', value=False)
                    merge_count = gr.Slider(minimum=2, maximum=20, value=2, step=1, label='Số câu mỗi file ghép')

                with gr.Column(visible=False) as batch_conv_only_col:
                    batch_conv_output_mode = gr.Radio(
                        choices=['Bình thường', 'Chỉ xuất file đã gộp'],
                        value='Bình thường',
                        label='Tùy chọn xuất (Batch/HSK Conversation)'
                    )

            with gr.Group(visible=True) as clone_group:
                gr.Markdown("### 🎤 Reference Audios")
                with gr.Row():
                    with gr.Column():
                        ref_audio_1 = gr.Audio(label='Clone 1 (A) - Audio', type='filepath')
                        ref_text_1 = gr.Textbox(label='Clone 1 - Transcript (Optional)', lines=1)
                    with gr.Column():
                        ref_audio_2 = gr.Audio(label='Clone 2 (B) - Audio', type='filepath')
                        ref_text_2 = gr.Textbox(label='Clone 2 - Transcript (Optional)', lines=1)
                    with gr.Column(visible=False) as clone_3_col:
                        ref_audio_3 = gr.Audio(label='Clone 3 (Người hỏi) - Audio', type='filepath')
                        ref_text_3 = gr.Textbox(label='Clone 3 - Transcript (Optional)', lines=1)

            design_preset = gr.Dropdown(choices=list(DESIGN_PRESETS.keys()), value='Female, British accent', label='Voice style preset', visible=False)
            custom_instruct = gr.Textbox(label='Custom voice instruction', visible=False)

            mode_radio.change(
                fn=update_ui_visibility,
                inputs=[mode_radio, clone_sub_mode],
                outputs=[clone_group, clone_sub_mode, line_settings_row, batch_only_col, batch_conv_only_col, design_preset, custom_instruct, clone_3_col]
            )
            clone_sub_mode.change(
                fn=update_ui_visibility,
                inputs=[mode_radio, clone_sub_mode],
                outputs=[clone_group, clone_sub_mode, line_settings_row, batch_only_col, batch_conv_only_col, design_preset, custom_instruct, clone_3_col]
            )

            design_preset.change(fn=update_preset, inputs=design_preset, outputs=custom_instruct)

            with gr.Row():
                steps_sl = gr.Slider(minimum=8, maximum=64, value=32, step=8, label='Diffusion steps')
                speed_sl = gr.Slider(minimum=0.5, maximum=2.0, value=1.0, step=0.1, label='Global Speed')
                pause_sl = gr.Slider(minimum=0.0, maximum=3.0, value=0.2, step=0.1, label='Pause between lines (s)')

            with gr.Row():
                audio_quality = gr.Dropdown(
                    choices=['High (24kHz - Original)', 'Medium (16kHz - Save 33%)', 'Low (8kHz - Save 66%)'],
                    value='High (24kHz - Original)',
                    label='Audio Quality / File Size'
                )
                audio_format = gr.Radio(
                    choices=['WAV', 'MP3'],
                    value='MP3',
                    label='Output Extension Format'
                )

            gen_btn  = gr.Button('Generate Speech', elem_classes='gen-btn')
            status_md = gr.Markdown('')

        with gr.Column(scale=2):
            audio_out = gr.Audio(label='Output Audio (Single Mode)', type='filepath', interactive=False)
            file_out = gr.File(label='Download ZIP (Batch/HSK Mode)', interactive=False)

            gr.Markdown('### 🛠 Format Optimization Guide')
            gr.Markdown(
                '- **WAV:** Lossless uncompressed format. Best quality but results in very large file sizes.\n'
                '- **MP3:** Compressed format. Drastically reduces disk storage space (up to 80-90% saving), perfect for sending files over mobile networks or saving local space.'
            )

    gen_btn.click(
        fn=generate,
        inputs=[
            text_input, mode_radio, clone_sub_mode, odd_voice, even_voice, naming_rule,
            enable_merge, merge_count, batch_conv_output_mode,
            ref_audio_1, ref_text_1, ref_audio_2, ref_text_2, ref_audio_3, ref_text_3,
            design_preset, custom_instruct,
            steps_sl, speed_sl, pause_sl, audio_quality, audio_format
        ],
        outputs=[audio_out, file_out, status_md]
    )

if __name__ == "__main__":
    demo.launch(share=True)

/tmp/ipykernel_4046/1710930833.py:474: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title='OmniVoice Ultra', theme=gr.themes.Soft(primary_hue='emerald', secondary_hue='blue'), css=css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d2ffe8624470b94c4e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
